## TD-Gammon

In [31]:
# board[i] > 0  means white has that many pieces on point i
# board[i] < 0  means black has that many pieces on point i
# board[i] = 0  means empty

board = [0] * 26

# index 0    = white's bar
# index 25   = black's bar
# index 1-24 = the 24 points

# Starting position
board = [0,                    # white bar
         -2, 0, 0, 0, 0, 5,   # points 1-6
          0, 3, 0, 0, 0, -5,  # points 7-12
          5, 0, 0, 0, -3, 0,  # points 13-18
         -5, 0, 0, 0, 0, 2,   # points 19-24
          0]                   # black bar

In [32]:
def flip_board(board):
    # Create a new board with the positions flipped
    new_board = [0] * 26 
    new_board[0] = board[25]
    new_board[25] = board[0]
    for i in range(1, 25):
        new_board[i] = -board[26 - i]
    return new_board

In [33]:
def encode_board(board, player, cube=1, cube_owner=None):
    inputs = []

    if player == "BLACK":
        board = flip_board(board)

    mine_on_board = 0
    opp_on_board  = 0

    for i in range(1, 25):
        n     = board[i]
        mine  = max(n, 0)
        inputs.append(1.0 if mine >= 1 else 0.0)
        inputs.append(1.0 if mine >= 2 else 0.0)
        inputs.append(1.0 if mine >= 3 else 0.0)
        inputs.append(max(mine - 3, 0) / 2.0)
        mine_on_board += mine

        theirs = max(-n, 0)
        inputs.append(1.0 if theirs >= 1 else 0.0)
        inputs.append(1.0 if theirs >= 2 else 0.0)
        inputs.append(1.0 if theirs >= 3 else 0.0)
        inputs.append(max(theirs - 3, 0) / 2.0)
        opp_on_board += theirs

    # board[0] = current player bar, board[25] = opponent bar (after flip)
    my_off  = 15 - mine_on_board - board[0]
    opp_off = 15 - opp_on_board  - board[25]

    inputs.append(board[0]  / 2.0)
    inputs.append(board[25] / 2.0)
    inputs.append(my_off    / 15.0)
    inputs.append(opp_off   / 15.0)
    inputs.append(1.0 if player == "WHITE" else 0.0)
    inputs.append(cube / 64.0)
    inputs.append(1.0 if cube_owner == player else 0.0)  # I own the cube
    inputs.append(1.0 if cube_owner is None   else 0.0)  # cube is centred

    return inputs  # 200 floats


In [34]:
import copy
import numpy as np
import random
import time
import torch
import torch.nn as nn

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

DOUBLE_THRESHOLD = 0.5   # double when raw equity exceeds this
DROP_THRESHOLD   = 0.75  # opponent drops when doubler's equity exceeds this

# All 21 distinct dice outcomes; non-doubles weight=2, doubles weight=1 (out of 36)
ALL_DICE = [(1,1),(1,2),(1,3),(1,4),(1,5),(1,6),
            (2,2),(2,3),(2,4),(2,5),(2,6),
            (3,3),(3,4),(3,5),(3,6),
            (4,4),(4,5),(4,6),
            (5,5),(5,6),
            (6,6)]


def initial_board():
    return [0, -2, 0, 0, 0, 0, 5, 0, 3, 0, 0, 0, -5, 5, 0, 0, 0, -3, 0, -5, 0, 0, 0, 0, 2, 0]


def roll_dice():
    d1, d2 = random.randint(1, 6), random.randint(1, 6)
    return [d1, d2, d1, d2] if d1 == d2 else [d1, d2]


def game_outcome(board):
    """Returns (winner, outcome_type) or (None, None). outcome_type: 'win'/'gammon'/'backgammon'."""
    white_on = sum(board[i] for i in range(1, 25) if board[i] > 0)
    black_on = sum(-board[i] for i in range(1, 25) if board[i] < 0)
    white_off = 15 - white_on - board[0]
    black_off = 15 - black_on - board[25]

    if white_off == 15:
        winner = "WHITE"
        loser_off, loser_bar = black_off, board[25]
        loser_in_winner_home = any(board[i] < 0 for i in range(1, 7))
    elif black_off == 15:
        winner = "BLACK"
        loser_off, loser_bar = white_off, board[0]
        loser_in_winner_home = any(board[i] > 0 for i in range(19, 25))
    else:
        return None, None

    if loser_off == 0 and (loser_bar > 0 or loser_in_winner_home):
        return winner, "backgammon"
    if loser_off == 0:
        return winner, "gammon"
    return winner, "win"


def outcome_target(winner, outcome_type, current_player):
    """6-element terminal target from current_player's perspective.
    [p_win, p_gammon_win, p_backgammon_win, p_loss, p_gammon_loss, p_backgammon_loss]"""
    t = np.zeros(6, dtype=np.float32)
    if winner == current_player:
        t[{"win": 0, "gammon": 1, "backgammon": 2}[outcome_type]] = 1.0
    else:
        t[{"win": 3, "gammon": 4, "backgammon": 5}[outcome_type]] = 1.0
    return t


class TDGammon(nn.Module):
    def __init__(self, input_size=200, hidden_size=160, output_size=6, lr=0.01):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, output_size),
            nn.Sigmoid()
        )
        self.to(device)
        self.optimizer = torch.optim.SGD(self.parameters(), lr=lr)

    def forward(self, x):
        return self.net(x)

    def td_update(self, v, delta):
        """
        v    : tensor output of forward(), must have gradient.
        delta: TD error — numpy array or tensor, treated as a constant (no grad).
        """
        if not isinstance(delta, torch.Tensor):
            delta = torch.tensor(delta, dtype=torch.float32, device=device)
        self.optimizer.zero_grad()
        loss = -(delta.detach() * v).sum()
        loss.backward()
        self.optimizer.step()


def compute_raw_equity(board, player, cube, cube_owner, net):
    """Returns probs · [1,2,3,-1,-2,-3] from player's perspective (no grad)."""
    x = torch.tensor(encode_board(board, player, cube, cube_owner),
                     dtype=torch.float32, device=device)
    with torch.no_grad():
        probs = net(x)
    weights = torch.tensor([1., 2., 3., -1., -2., -3.], device=device)
    return (probs * weights).sum().item()


# ---------------------------------------------------------------------------
# Move generator — mirrors Swift MoveGenerator exactly.
# board[0]=white bar, board[25]=black bar, board[1-24]: +white/-black
# Move steps: (from, to, die)  |  -1=white bears off, 26=black bears off
# ---------------------------------------------------------------------------

def generate_legal_moves(board, dice, player):
    results = []
    _gen(board, list(dice), player, [], results)

    if not results:
        return []
    max_used = max(len(m) for m, _ in results)
    if max_used == 0:
        return []

    candidates = [(m, s) for m, s in results if len(m) == max_used]

    if max_used == 1 and len(dice) == 2 and dice[0] != dice[1]:
        higher = max(dice)
        with_higher = [(m, s) for m, s in candidates if m[0][2] == higher]
        if with_higher:
            candidates = with_higher

    seen, unique = set(), []
    for m, s in candidates:
        key = tuple(s)
        if key not in seen:
            seen.add(key)
            unique.append(m)
    return unique


def _gen(board, remaining, player, current, results):
    singles = _singles(board, remaining, player)
    if not singles:
        results.append((current, board)); return
    tried = set()
    for die in remaining:
        if die in tried: continue
        tried.add(die)
        for m in [s for s in singles if s[2] == die]:
            nxt = list(remaining); nxt.remove(die)
            _gen(_apply_step(board, m, player), nxt, player, current + [m], results)


def _singles(board, dice, player):
    moves = []
    for die in set(dice):
        if player == "WHITE":
            if board[0] > 0:
                to = 25 - die
                if board[to] > -2:
                    moves.append((0, to, die))
            else:
                for pt in range(1, 25):
                    if board[pt] <= 0: continue
                    dest = pt - die
                    if 1 <= dest <= 24:
                        if board[dest] > -2: moves.append((pt, dest, die))
                    elif _can_bear_off_white(board) and _bear_off_ok_white(pt, dest, board):
                        moves.append((pt, -1, die))
        else:
            if board[25] > 0:
                to = die
                if board[to] < 2:
                    moves.append((25, to, die))
            else:
                for pt in range(1, 25):
                    if board[pt] >= 0: continue
                    dest = pt + die
                    if 1 <= dest <= 24:
                        if board[dest] < 2: moves.append((pt, dest, die))
                    elif _can_bear_off_black(board) and _bear_off_ok_black(pt, dest, board):
                        moves.append((pt, 26, die))
    return moves


def _can_bear_off_white(board):
    return board[0] == 0 and all(board[i] <= 0 for i in range(7, 25))

def _can_bear_off_black(board):
    return board[25] == 0 and all(board[i] >= 0 for i in range(1, 19))

def _bear_off_ok_white(point, dest, board):
    if dest == 0: return True
    return all(board[p] <= 0 for p in range(point + 1, 7))

def _bear_off_ok_black(point, dest, board):
    if dest == 25: return True
    return all(board[p] >= 0 for p in range(19, point))


def _apply_step(board, step, player):
    board = list(board)
    frm, to, _ = step
    if player == "WHITE":
        if frm == 0: board[0] -= 1
        else:        board[frm] -= 1
        if to != -1:
            if board[to] == -1: board[to] = 0; board[25] += 1
            board[to] += 1
    else:
        if frm == 25: board[25] -= 1
        else:         board[frm] += 1
        if to != 26:
            if board[to] == 1: board[to] = 0; board[0] += 1
            board[to] -= 1
    return board


def apply_move(board, move, player):
    for step in move:
        board = _apply_step(board, step, player)
    return board


_EQUITY_WEIGHTS = None  # initialised on first call, avoids repeated tensor creation

def eval_equity(board, player, network, cube, cube_owner):
    """Encode board and return scalar equity from player's perspective."""
    global _EQUITY_WEIGHTS
    if _EQUITY_WEIGHTS is None:
        _EQUITY_WEIGHTS = torch.tensor([1., 2., 3., -1., -2., -3.], device=device)
    x     = torch.tensor(encode_board(board, player, cube, cube_owner),
                         dtype=torch.float32, device=device)
    probs = network(x)
    return (probs * _EQUITY_WEIGHTS).sum().item()


# ---------------------------------------------------------------------------
# choose_move_1ply  — used during training (fast, already batched)
# choose_move       — used during inference (3-ply, fully batched)
# ---------------------------------------------------------------------------

def choose_move_1ply(board, dice, player, network, cube=1, cube_owner=None):
    """1-ply move selection: single batched forward pass over all legal moves."""
    legal_moves = generate_legal_moves(board, dice, player)
    if not legal_moves:
        return None

    inputs = [encode_board(apply_move(board, m, player), player, cube, cube_owner)
              for m in legal_moves]
    batch  = torch.tensor(inputs, dtype=torch.float32, device=device)

    with torch.no_grad():
        probs = network(batch)

    weights  = torch.tensor([1., 2., 3., -1., -2., -3.], device=device)
    equities = (probs * weights).sum(dim=1)
    return legal_moves[equities.argmax().item()]


def choose_move(board, dice, player, network, cube=1, cube_owner=None):
    """3-ply move selection with forward pruning.

    Batching strategy — forward passes per call:
      Ply 1 : 1 pass  (all K candidates)
      Ply 2 : 1 pass  per candidate  (all 21 dice × opp_moves stacked)
      Ply 3 : 1 pass  per (candidate × ply-2 dice outcome)
                       (all J opp_boards × 21 dice × your_moves stacked)
    Total   : 1 + K×(1 + 21) ≈ 112 passes  vs  ~8 400 in the sequential version.
    """
    legal_moves = generate_legal_moves(board, dice, player)
    if not legal_moves:
        return None

    opponent = "BLACK" if player == "WHITE" else "WHITE"

    global _EQUITY_WEIGHTS
    if _EQUITY_WEIGHTS is None:
        _EQUITY_WEIGHTS = torch.tensor([1., 2., 3., -1., -2., -3.], device=device)
    W = _EQUITY_WEIGHTS

    with torch.no_grad():

        def batch_eval(enc_list):
            """Single forward pass over a list of pre-encoded board vectors."""
            t = torch.tensor(enc_list, dtype=torch.float32, device=device)
            return (network(t) * W).sum(dim=1).tolist()

        # ── Ply 1: batch-evaluate all candidates (1 forward pass) ──────────
        K = min(len(legal_moves), 5)
        your_boards = [apply_move(board, m, player) for m in legal_moves]
        eq_1ply     = batch_eval([encode_board(b, player, cube, cube_owner) for b in your_boards])
        top_k_idx   = sorted(range(len(legal_moves)), key=lambda i: eq_1ply[i], reverse=True)[:K]
        candidates  = [(legal_moves[i], your_boards[i]) for i in top_k_idx]

        candidate_equities = []

        for _, your_board in candidates:

            # ── Ply 2: collect every opp board across all 21 dice; 1 forward pass ──
            ply2_enc  = []
            ply2_meta = []   # None  |  (weight, start, end, opp_moves)

            for d1, d2 in ALL_DICE:
                w2            = 1 if d1 == d2 else 2
                opp_dice_list = [d1, d2, d1, d2] if d1 == d2 else [d1, d2]
                opp_moves     = generate_legal_moves(your_board, opp_dice_list, opponent)
                if not opp_moves:
                    ply2_meta.append(None)
                    continue
                s = len(ply2_enc)
                for m in opp_moves:
                    ply2_enc.append(encode_board(apply_move(your_board, m, opponent),
                                                 opponent, cube, cube_owner))
                ply2_meta.append((w2, s, len(ply2_enc), opp_moves))

            if not ply2_enc:
                candidate_equities.append(
                    batch_eval([encode_board(your_board, player, cube, cube_owner)])[0])
                continue

            ply2_eqs = batch_eval(ply2_enc)

            ply2_w_sum = 0.0
            ply2_w_tot = 0

            for meta2 in ply2_meta:
                if meta2 is None:
                    continue
                w2, s2, e2, opp_moves = meta2

                # Prune to J=3 best opponent moves for this dice outcome
                opp_slice = ply2_eqs[s2:e2]
                J      = min(len(opp_moves), 3)
                top_j  = sorted(range(len(opp_moves)),
                                 key=lambda i: opp_slice[i], reverse=True)[:J]
                pruned = [apply_move(your_board, opp_moves[i], opponent) for i in top_j]

                # ── Ply 3: batch all J opp boards × 21 dice × your_moves; 1 forward pass ──
                ply3_enc  = []
                ply3_meta = []   # None  |  (opp_idx, weight, start, end)

                for opp_idx, opp_board in enumerate(pruned):
                    for d1_3, d2_3 in ALL_DICE:
                        w3           = 1 if d1_3 == d2_3 else 2
                        your_dice_3  = [d1_3, d2_3, d1_3, d2_3] if d1_3 == d2_3 else [d1_3, d2_3]
                        your_moves_3 = generate_legal_moves(opp_board, your_dice_3, player)
                        if not your_moves_3:
                            ply3_meta.append(None)
                            continue
                        s3 = len(ply3_enc)
                        for m in your_moves_3:
                            ply3_enc.append(encode_board(apply_move(opp_board, m, player),
                                                         player, cube, cube_owner))
                        ply3_meta.append((opp_idx, w3, s3, len(ply3_enc)))

                if ply3_enc:
                    ply3_eqs  = batch_eval(ply3_enc)
                    ply3_sums = [0.0] * len(pruned)
                    ply3_tots = [0.0] * len(pruned)
                    for meta3 in ply3_meta:
                        if meta3 is None:
                            continue
                        opp_idx, w3, s3, e3 = meta3
                        ply3_sums[opp_idx] += w3 * max(ply3_eqs[s3:e3])
                        ply3_tots[opp_idx] += w3
                    opp_equities = [
                        ply3_sums[i] / ply3_tots[i] if ply3_tots[i] > 0 else 0.0
                        for i in range(len(pruned))
                    ]
                else:
                    opp_equities = [0.0] * len(pruned)

                ply2_w_sum += w2 * min(opp_equities)
                ply2_w_tot += w2

            eq_candidate = ply2_w_sum / ply2_w_tot if ply2_w_tot > 0 else 0.0
            candidate_equities.append(eq_candidate)

        best_idx = max(range(len(candidates)), key=lambda i: candidate_equities[i])
        return candidates[best_idx][0]


def win_rate_vs_snapshot(net, snapshot, n_games=1000):
    """Play current net (WHITE) vs a previous snapshot (BLACK), both cube-aware.
    Uses 1-ply move selection for speed. Returns WHITE win rate 0.0-1.0."""
    net.eval()
    snapshot.eval()
    wins = 0
    with torch.no_grad():
        for _ in range(n_games):
            board      = initial_board()
            player     = "WHITE"
            cube_value = 1
            cube_owner = None

            for _ in range(10_000):
                opponent    = "BLACK" if player == "WHITE" else "WHITE"
                current_net = net if player == "WHITE" else snapshot

                # Cube decision
                can_double = (cube_owner is None or cube_owner == player) and cube_value < 64
                if can_double:
                    eq = compute_raw_equity(board, player, cube_value, cube_owner, current_net)
                    if eq > DROP_THRESHOLD:
                        if player == "WHITE":
                            wins += 1
                        break
                    elif eq > DOUBLE_THRESHOLD:
                        cube_value *= 2
                        cube_owner  = opponent

                dice  = roll_dice()
                moves = generate_legal_moves(board, dice, player)
                if moves:
                    move  = choose_move_1ply(board, dice, player, current_net, cube_value, cube_owner)
                    board = apply_move(board, move, player)

                winner, _ = game_outcome(board)
                if winner is not None:
                    if winner == "WHITE":
                        wins += 1
                    break
                player = opponent

    net.train()
    return wins / n_games


def train(n_games=100_000, lr=0.01, hidden_size=160, print_every=1_000,
          eval_every=10_000, snapshot_every=5_000):
    """
    Cube-aware self-play TD training on MPS (Apple GPU).
    Uses 1-ply move selection for speed during training.
    Eval compares current net vs a snapshot taken snapshot_every games ago.
    """
    net      = TDGammon(hidden_size=hidden_size, lr=lr)
    snapshot = copy.deepcopy(net)   # refreshed every snapshot_every games
    t0       = time.time()

    # Pre-create equity weight tensor once
    global _EQUITY_WEIGHTS
    if _EQUITY_WEIGHTS is None:
        _EQUITY_WEIGHTS = torch.tensor([1., 2., 3., -1., -2., -3.], device=device)
    W = _EQUITY_WEIGHTS

    print(f"{'Game':>10}  {'Progress':>8}  {'Games/sec':>10}  {'vs Snapshot':>12}")
    print("-" * 50)

    for game in range(n_games):
        board      = initial_board()
        player     = "WHITE"
        cube_value = 1
        cube_owner = None   # None = centred; "WHITE"/"BLACK" = that side owns it

        for _ in range(10_000):
            opponent = "BLACK" if player == "WHITE" else "WHITE"

            # === BOARD VALUE — computed once; reused for cube equity and TD update ===
            x = torch.tensor(encode_board(board, player, cube_value, cube_owner),
                             dtype=torch.float32, device=device)
            v = net(x)

            # === CUBE DECISION — derive equity from v (no extra forward pass) ===
            can_double = (cube_owner is None or cube_owner == player) and cube_value < 64
            if can_double:
                eq = (v.detach() * W).sum().item()
                if eq > DOUBLE_THRESHOLD:
                    if eq > DROP_THRESHOLD:
                        target = torch.tensor([1., 0., 0., 0., 0., 0.], device=device)
                        net.td_update(v, target - v.detach())
                        break
                    cube_value *= 2
                    cube_owner  = opponent
                    # Cube state changed → recompute x and v with new cube encoding
                    x = torch.tensor(encode_board(board, player, cube_value, cube_owner),
                                     dtype=torch.float32, device=device)
                    v = net(x)

            # === ROLL AND MOVE (1-ply for training speed) ===
            dice  = roll_dice()
            moves = generate_legal_moves(board, dice, player)
            if moves:
                board = apply_move(
                    board,
                    choose_move_1ply(board, dice, player, net, cube_value, cube_owner),
                    player
                )

            # === TERMINAL CHECK ===
            winner, outcome = game_outcome(board)
            if winner is not None:
                target = torch.tensor(outcome_target(winner, outcome, player), device=device)
                net.td_update(v, target - v.detach())
                break

            # === TD UPDATE ===
            next_player = opponent
            with torch.no_grad():
                v_new = net(torch.tensor(
                    encode_board(board, next_player, cube_value, cube_owner),
                    dtype=torch.float32, device=device))
            net.td_update(v, -v_new - v.detach())
            player = next_player

        # Advance snapshot independently of eval
        if (game + 1) % snapshot_every == 0:
            snapshot = copy.deepcopy(net)

        if (game + 1) % print_every == 0:
            elapsed = time.time() - t0
            gps     = (game + 1) / elapsed
            pct     = (game + 1) / n_games * 100

            if (game + 1) % eval_every == 0:
                wr = win_rate_vs_snapshot(net, snapshot)
                print(f"{game+1:>10,}  {pct:>7.1f}%  {gps:>10.1f}  {wr:>11.1%}")
            else:
                print(f"{game+1:>10,}  {pct:>7.1f}%  {gps:>10.1f}  {'—':>12}")

    print("-" * 50)
    print(f"Done. Total time: {(time.time()-t0)/60:.1f} min")
    return net


Using device: mps


In [35]:
import multiprocessing as mp
import queue
import sys

# ---------------------------------------------------------------------------
# bg_worker.py source — written to disk so spawned processes can import it.
# Contains all CPU-only game logic; no MPS, no optimizer, no autograd.
# ---------------------------------------------------------------------------

_BG_WORKER_SRC = """
import random
import traceback
import numpy as np
import torch
import torch.nn as nn

DOUBLE_THRESHOLD = 0.5
DROP_THRESHOLD   = 0.75
_W = np.array([1., 2., 3., -1., -2., -3.], dtype=np.float32)
_SWAP = np.array([3, 4, 5, 0, 1, 2])


def initial_board():
    return [0, -2, 0, 0, 0, 0, 5, 0, 3, 0, 0, 0, -5, 5, 0, 0, 0, -3, 0, -5, 0, 0, 0, 0, 2, 0]


def roll_dice():
    d1, d2 = random.randint(1, 6), random.randint(1, 6)
    return [d1, d2, d1, d2] if d1 == d2 else [d1, d2]


def flip_board(board):
    new_board = [0] * 26
    new_board[0] = board[25]
    new_board[25] = board[0]
    for i in range(1, 25):
        new_board[i] = -board[26 - i]
    return new_board


def encode_board(board, player, cube=1, cube_owner=None):
    inputs = []
    if player == "BLACK":
        board = flip_board(board)
    mine_on_board = 0
    opp_on_board  = 0
    for i in range(1, 25):
        n    = board[i]
        mine = max(n, 0)
        inputs.append(1.0 if mine >= 1 else 0.0)
        inputs.append(1.0 if mine >= 2 else 0.0)
        inputs.append(1.0 if mine >= 3 else 0.0)
        inputs.append(max(mine - 3, 0) / 2.0)
        mine_on_board += mine
        theirs = max(-n, 0)
        inputs.append(1.0 if theirs >= 1 else 0.0)
        inputs.append(1.0 if theirs >= 2 else 0.0)
        inputs.append(1.0 if theirs >= 3 else 0.0)
        inputs.append(max(theirs - 3, 0) / 2.0)
        opp_on_board += theirs
    my_off  = 15 - mine_on_board - board[0]
    opp_off = 15 - opp_on_board  - board[25]
    inputs.append(board[0]  / 2.0)
    inputs.append(board[25] / 2.0)
    inputs.append(my_off    / 15.0)
    inputs.append(opp_off   / 15.0)
    inputs.append(1.0 if player == "WHITE" else 0.0)
    inputs.append(cube / 64.0)
    inputs.append(1.0 if cube_owner == player else 0.0)
    inputs.append(1.0 if cube_owner is None   else 0.0)
    return inputs  # 200 floats


def game_outcome(board):
    white_on  = sum(board[i] for i in range(1, 25) if board[i] > 0)
    black_on  = sum(-board[i] for i in range(1, 25) if board[i] < 0)
    white_off = 15 - white_on - board[0]
    black_off = 15 - black_on - board[25]
    if white_off == 15:
        winner = "WHITE"
        loser_off, loser_bar = black_off, board[25]
        loser_in_winner_home = any(board[i] < 0 for i in range(1, 7))
    elif black_off == 15:
        winner = "BLACK"
        loser_off, loser_bar = white_off, board[0]
        loser_in_winner_home = any(board[i] > 0 for i in range(19, 25))
    else:
        return None, None
    if loser_off == 0 and (loser_bar > 0 or loser_in_winner_home):
        return winner, "backgammon"
    if loser_off == 0:
        return winner, "gammon"
    return winner, "win"


def outcome_target(winner, outcome_type, current_player):
    t = np.zeros(6, dtype=np.float32)
    if winner == current_player:
        t[{"win": 0, "gammon": 1, "backgammon": 2}[outcome_type]] = 1.0
    else:
        t[{"win": 3, "gammon": 4, "backgammon": 5}[outcome_type]] = 1.0
    return t


def generate_legal_moves(board, dice, player):
    results = []
    _gen(board, list(dice), player, [], results)
    if not results:
        return []
    max_used = max(len(m) for m, _ in results)
    if max_used == 0:
        return []
    candidates = [(m, s) for m, s in results if len(m) == max_used]
    if max_used == 1 and len(dice) == 2 and dice[0] != dice[1]:
        higher = max(dice)
        with_higher = [(m, s) for m, s in candidates if m[0][2] == higher]
        if with_higher:
            candidates = with_higher
    seen, unique = set(), []
    for m, s in candidates:
        key = tuple(s)
        if key not in seen:
            seen.add(key)
            unique.append(m)
    return unique


def _gen(board, remaining, player, current, results):
    singles = _singles(board, remaining, player)
    if not singles:
        results.append((current, board)); return
    tried = set()
    for die in remaining:
        if die in tried: continue
        tried.add(die)
        for m in [s for s in singles if s[2] == die]:
            nxt = list(remaining); nxt.remove(die)
            _gen(_apply_step(board, m, player), nxt, player, current + [m], results)


def _singles(board, dice, player):
    moves = []
    for die in set(dice):
        if player == "WHITE":
            if board[0] > 0:
                to = 25 - die
                if board[to] > -2: moves.append((0, to, die))
            else:
                for pt in range(1, 25):
                    if board[pt] <= 0: continue
                    dest = pt - die
                    if 1 <= dest <= 24:
                        if board[dest] > -2: moves.append((pt, dest, die))
                    elif _can_bear_off_white(board) and _bear_off_ok_white(pt, dest, board):
                        moves.append((pt, -1, die))
        else:
            if board[25] > 0:
                to = die
                if board[to] < 2: moves.append((25, to, die))
            else:
                for pt in range(1, 25):
                    if board[pt] >= 0: continue
                    dest = pt + die
                    if 1 <= dest <= 24:
                        if board[dest] < 2: moves.append((pt, dest, die))
                    elif _can_bear_off_black(board) and _bear_off_ok_black(pt, dest, board):
                        moves.append((pt, 26, die))
    return moves


def _can_bear_off_white(board):
    return board[0] == 0 and all(board[i] <= 0 for i in range(7, 25))

def _can_bear_off_black(board):
    return board[25] == 0 and all(board[i] >= 0 for i in range(1, 19))

def _bear_off_ok_white(point, dest, board):
    if dest == 0: return True
    return all(board[p] <= 0 for p in range(point + 1, 7))

def _bear_off_ok_black(point, dest, board):
    if dest == 25: return True
    return all(board[p] >= 0 for p in range(19, point))

def _apply_step(board, step, player):
    board = list(board)
    frm, to, _ = step
    if player == "WHITE":
        if frm == 0: board[0] -= 1
        else:        board[frm] -= 1
        if to != -1:
            if board[to] == -1: board[to] = 0; board[25] += 1
            board[to] += 1
    else:
        if frm == 25: board[25] -= 1
        else:         board[frm] += 1
        if to != 26:
            if board[to] == 1: board[to] = 0; board[0] += 1
            board[to] -= 1
    return board

def apply_move(board, move, player):
    for step in move:
        board = _apply_step(board, step, player)
    return board


class _WorkerNet(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # Attribute named 'net' so state_dict keys match TDGammon exactly.
        self.net = nn.Sequential(
            nn.Linear(200, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, 6),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)


def _choose_1ply_cpu(board, dice, player, net, cube, cube_owner):
    legal_moves = generate_legal_moves(board, dice, player)
    if not legal_moves:
        return None
    opp = "BLACK" if player == "WHITE" else "WHITE"
    encodings = [encode_board(apply_move(board, m, player), opp, cube, cube_owner)
                 for m in legal_moves]
    batch = torch.tensor(encodings, dtype=torch.float32)
    with torch.no_grad():
        opp_eq = (net(batch).numpy() * _W).sum(axis=1)   # opponent's equity
    return legal_moves[int(opp_eq.argmin())]             # minimize it


def _play_games(net, n_batch):
    experience = []
    with torch.no_grad():
        for _ in range(n_batch):
            board      = initial_board()
            player     = "WHITE"
            cube_value = 1
            cube_owner = None
            for _ in range(10_000):
                opponent = "BLACK" if player == "WHITE" else "WHITE"
                enc  = np.array(encode_board(board, player, cube_value, cube_owner), dtype=np.float32)
                v_np = net(torch.from_numpy(enc)).numpy().copy()
                dice = roll_dice()
                best = _choose_1ply_cpu(board, dice, player, net, cube_value, cube_owner)
                if best is not None:
                    board = apply_move(board, best, player)
                winner, outcome = game_outcome(board)
                if winner is not None:
                    target = outcome_target(winner, outcome, player)
                    experience.append((enc, target - v_np))
                    break
                next_player = opponent
                next_enc = np.array(encode_board(board, next_player, cube_value, cube_owner), dtype=np.float32)
                v_new_np = net(torch.from_numpy(next_enc)).numpy().copy()
                target = v_new_np[_SWAP]          # opponent's loss probs are my win probs
                experience.append((enc, target - v_np))
                player = next_player
    return experience


def worker_fn(worker_id, task_queue, result_queue, hidden_size):
    net = _WorkerNet(hidden_size)
    net.eval()
    try:
        while True:
            msg = task_queue.get()
            if msg is None:
                break
            n_batch, state_dict_np = msg
            net.load_state_dict(
                {k: torch.tensor(v, dtype=torch.float32) for k, v in state_dict_np.items()}
            )
            experience = _play_games(net, n_batch)
            result_queue.put((worker_id, experience))
    except Exception:
        result_queue.put((worker_id, RuntimeError(traceback.format_exc())))
"""


def _write_bg_worker():
    with open('bg_worker.py', 'w') as f:
        f.write(_BG_WORKER_SRC)


# ---------------------------------------------------------------------------
# train_parallel — multi-process TD training.
#
# Workers (CPU-only):  play n_batch games, collect (enc, delta) pairs, return.
# Main (MPS/GPU):      replay each (enc, delta) through the master net, sync weights.
# ---------------------------------------------------------------------------

def train_parallel(n_games=1_500_000, lr=0.01, hidden_size=160,
                   print_every=10_000, eval_every=100_000, snapshot_every=100_000,
                   n_workers=6, n_batch=10):

    _write_bg_worker()

    if 'bg_worker' in sys.modules:
        del sys.modules['bg_worker']
    import bg_worker

    try:
        mp.set_start_method('spawn', force=True)
    except RuntimeError:
        pass

    net      = TDGammon(hidden_size=hidden_size, lr=lr)
    snapshot = copy.deepcopy(net)
    t0       = time.time()

    def get_weights():
        return {k: v.cpu().numpy() for k, v in net.state_dict().items()}

    result_queue = mp.Queue()
    task_queues  = [mp.Queue() for _ in range(n_workers)]

    def spawn_worker(wid):
        p = mp.Process(
            target=bg_worker.worker_fn,
            args=(wid, task_queues[wid], result_queue, hidden_size),
            daemon=True,
        )
        p.start()
        return p

    initial_weights = get_weights()
    workers = [spawn_worker(wid) for wid in range(n_workers)]
    for wid, q in enumerate(task_queues):
        q.put((n_batch, initial_weights))

    total_games = 0
    print(f"{'Game':>10}  {'Progress':>8}  {'Games/sec':>10}  {'vs Snapshot':>12}")
    print("-" * 50)

    while total_games < n_games:
        try:
            wid, payload = result_queue.get(timeout=120)
        except queue.Empty:
            # 2-minute timeout: check for dead workers and restart them
            for i, p in enumerate(workers):
                if not p.is_alive():
                    print(f"Worker {i} died unexpectedly — restarting.")
                    workers[i] = spawn_worker(i)
                    task_queues[i].put((n_batch, get_weights()))
            continue

        if isinstance(payload, Exception):
            print(f"Worker {wid} error:\n{payload}")
            workers[wid].terminate()
            workers[wid] = spawn_worker(wid)
            task_queues[wid].put((n_batch, get_weights()))
            continue

        # Replay worker experience through master net on MPS
        net.train()
        for enc, delta in payload:
            x = torch.tensor(enc,   dtype=torch.float32, device=device)
            d = torch.tensor(delta, dtype=torch.float32, device=device)
            v = net(x)
            net.td_update(v, d)

        # Send fresh weights back to this worker immediately
        task_queues[wid].put((n_batch, get_weights()))

        prev_games   = total_games
        total_games += n_batch

        if total_games // snapshot_every > prev_games // snapshot_every:
            snapshot = copy.deepcopy(net)

        if total_games // print_every > prev_games // print_every:
            elapsed = time.time() - t0
            gps     = total_games / elapsed
            pct     = min(total_games / n_games * 100, 100.0)
            if total_games // eval_every > prev_games // eval_every:
                wr = win_rate_vs_snapshot(net, snapshot)
                print(f"{total_games:>10,}  {pct:>7.1f}%  {gps:>10.1f}  {wr:>11.1%}")
            else:
                print(f"{total_games:>10,}  {pct:>7.1f}%  {gps:>10.1f}  {'—':>12}")

    for q in task_queues:
        q.put(None)
    for p in workers:
        p.join(timeout=5)
        if p.is_alive():
            p.terminate()

    print("-" * 50)
    print(f"Done. Total time: {(time.time()-t0)/60:.1f} min")
    return net


In [40]:
if __name__ == '__main__':
    td_gammon = train_parallel(
        n_games        = 10_000,
        lr             = 0.01,
        hidden_size    = 160,
        print_every    = 1_000,
        eval_every     = 5_000,
        snapshot_every = 5_000,
        n_workers      = 8,
        n_batch        = 50,
    )

      Game  Progress   Games/sec   vs Snapshot
--------------------------------------------------
     1,000     10.0%        10.0             —


Process Process-24:
Process Process-22:
Process Process-20:
Process Process-19:
Process Process-25:
Process Process-26:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/ovunc/Desktop/Code_Related/Backgammon Teacher/bg_worker.py", line 277, in worker_fn
    experience = _play_games(net, n_batch)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ovunc/Desktop/Code_Related/Backgammon Teacher/bg_worker.py", line 249, in _play_games
    best = _choose_1ply_cpu(board, dice, player, net, cube_value, cube_owner)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ovunc/Desktop/Code_Related/Backgammon Teacher/bg_worker.py", line 219, in _choose_1ply_cpu
    with torch.no_grad():
  File "/Users/ovunc/Desktop/Code_R

KeyboardInterrupt: 

In [ ]:
torch.save(td_gammon.state_dict(), "tdgammon.pt")
print("Model saved to tdgammon.pt")

Model saved to tdgammon.pt


In [ ]:
import coremltools as ct

# Reload from disk to ensure clean CPU model (MPS tensors can't be traced)
class TDGammonCPU(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(200, 160),
            nn.Sigmoid(),
            nn.Linear(160, 6),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

cpu_model = TDGammonCPU()
cpu_model.load_state_dict(torch.load("tdgammon.pt", map_location="cpu"))
cpu_model.eval()

traced = torch.jit.trace(cpu_model.net, torch.zeros(1, 200))

ml_model = ct.convert(
    traced,
    inputs=[ct.TensorType(name="x", shape=(1, 200))],
    outputs=[ct.TensorType(name="output")]
)
ml_model.save("TDGammon.mlpackage")
print("Saved TDGammon.mlpackage")

When both 'convert_to' and 'minimum_deployment_target' not specified, 'convert_to' is set to "mlprogram" and 'minimum_deployment_target' is set to ct.target.iOS15 (which is same as ct.target.macOS12). Note: the model will not run on systems older than iOS15/macOS12/watchOS8/tvOS15. In order to make your model run on older system, please set the 'minimum_deployment_target' to iOS14/iOS13. Details please see the link: https://apple.github.io/coremltools/docs-guides/source/target-conversion-formats.html
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 12219.39 passes/s]

Saved TDGammon.mlpackage
